In [0]:
import pandas as pd
import math

In [0]:
spark.sql("use catalog proyecto_final_prueba")

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_ubicacion"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver


In [0]:
dim = pd.DataFrame(df_silver[["latitude", "longitude"]]).drop_duplicates().reset_index(drop=True)
dim["id_ubicacion"] = dim.index+1
catalogo_ciudades = {
    (-6.2294, -77.8728): "Chachapoyas", (-9.5278, -77.5278): "Huaraz",
    (-13.6339, -72.8814): "Abancay", (-16.3989, -71.5350): "Arequipa",
    (-13.1588, -74.2239): "Ayacucho", (-7.1638, -78.5003): "Cajamarca",
    (-12.0566, -77.1181): "Callao", (-13.5226, -71.9673): "Cusco",
    (-12.7826, -74.9727): "Huancavelica", (-9.9306, -76.2422): "Huánuco",
    (-14.0678, -75.7286): "Ica", (-12.0651, -75.2049): "Huancayo",
    (-8.1159, -79.0300): "Trujillo", (-6.7714, -79.8409): "Chiclayo",
    (-12.0432, -77.0282): "Lima", (-3.7491, -73.2538): "Iquitos",
    (-12.5933, -69.1836): "Puerto Maldonado", (-17.1983, -70.9357): "Moquegua",
    (-10.6675, -76.2567): "Cerro de Pasco", (-5.1945, -80.6328): "Piura",
    (-15.8402, -70.0219): "Puno", (-6.0333, -76.9667): "Moyobamba",
    (-18.0146, -70.2536): "Tacna", (-3.5669, -80.4515): "Tumbes",
    (-8.3791, -74.5539): "Pucallpa"
}

# 3. Función para encontrar la ciudad matemáticamente más cercana
def obtener_ciudad_cercana(lat_api, lon_api):
    ciudad_mas_cercana = "Desconocido"
    distancia_minima = float('inf') # Infinito inicial
    
    for (lat_cat, lon_cat), nombre_ciudad in catalogo_ciudades.items():
        # Distancia euclidiana simple (teorema de Pitágoras)
        distancia = math.sqrt((lat_api - lat_cat)**2 + (lon_api - lon_cat)**2)
        
        if distancia < distancia_minima:
            distancia_minima = distancia
            ciudad_mas_cercana = nombre_ciudad
            
    return ciudad_mas_cercana

# 4. Aplicamos la nueva función a cada fila
dim['nombre_lugar'] = dim.apply(lambda row: obtener_ciudad_cercana(row['latitude'], row['longitude']), axis=1)
columns = ["id_ubicacion", "latitude", "longitude", "nombre_lugar"]
dim = dim[columns]
dim

In [0]:
df_spark = spark.createDataFrame(dim)
df_spark.display()

In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")